# Imports

* First, turn on the fan, clean the robot (Live tumor only). Clean the shaking ring and the plastic insert it sits in especially carefully.
* Generally, you can follow the notebook and execute all cells from top to bottom to pick and place cuboids.

In [1]:
import sys
import os

# Add the upstream directory to sys.path
upstream_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if upstream_dir not in sys.path:
    sys.path.insert(0, upstream_dir)

from opentrons_api import ot2_api
from microtissue_manipulator import core, utils, camera
import numpy as np 
import cv2
import time
import json
import keyboard
import paths
import matplotlib.pyplot as plt
import requests
import datetime
import threading
import queue
import string
import pandas as pd
import multiprocessing as mp
from dataclasses import dataclass, fields, asdict, MISSING
from typing import get_type_hints, get_origin, get_args, Tuple, List, Dict, Any, Union
from ultralytics import YOLO
from enum import Enum

# Start the cameras

* Make sure the cameras are plugged into the computer.
* Execute following cell to start the cameras.

* If cameras are not working, unplug and plug back in. Also try restarting the notebook. 

In [2]:
cam_manager = camera.CameraManagerWindows()
print("Connected cameras:")
print(cam_manager.list_devices())

under_cam = camera.open_capture('underview_cam', cam_manager=cam_manager, resolution = (4000,3000), focus = 900)
over_cam = camera.open_capture('overview_cam_2', cam_manager=cam_manager)
frame_ops = camera.frameOperations(*over_cam.shape[0:-1])

calibration_profile = 'checkerboard'
frame_ops.load_camera_intrinsics(config_profile=calibration_profile, use_new_cam_mtx=True)

time.sleep(3)
ret, _ = under_cam.read()
if ret:
    print("Under camera is working correctly.")
else:
    print("Under camera is not working.")

ret, _ = over_cam.read()
if ret:
    print("Over camera is working correctly.")
else:
    print("Over camera is not working.")

Connected cameras:
['Arducam B0478 (USB3 48MP)', 'HD USB CAMERA']
Under camera is working correctly.
Over camera is working correctly.


# Initialization 

* Connect robot to the computer, robot setup

In [3]:
# Connect the robot to the computer and this notebook
openapi = ot2_api.OpentronsAPI()

In [4]:
# Set the offset if the platform is used
openapi.add_slot_offsets([5,8,9], (0,0,64.2))

In [6]:
# Use the light control to see if the robot is connected as a sanity check
openapi.toggle_lights()

<Response [200]>

In [7]:
# Always do once after robot was just turned on
openapi.home_robot()

Request status:
<Response [200]>
{
  "message": "Homing robot."
}


<Response [200]>

In [8]:
# Do after first launch
openapi.create_run()

Request status:
<Response [201]>
{
  "data": {
    "id": "34df5590-624b-4845-b79c-a0736ee2b40d",
    "ok": true,
    "createdAt": "2025-06-06T05:45:04.495966Z",
    "status": "idle",
    "current": true,
    "actions": [],
    "errors": [],
    "hasEverEnteredErrorRecovery": false,
    "pipettes": [],
    "modules": [],
    "labware": [],
    "liquids": [],
    "liquidClasses": [],
    "labwareOffsets": [],
    "runTimeParameters": [],
    "outputFileIds": []
  }
}


<Response [201]>

In [9]:
# Let the robot know that it has the P300 pipettor
openapi.load_pipette()

Request status:
<Response [201]>
{
  "data": {
    "id": "5660c734-f85e-4dd9-9301-7c8cc94f57e8",
    "createdAt": "2025-06-06T05:45:12.427202Z",
    "commandType": "loadPipette",
    "key": "5660c734-f85e-4dd9-9301-7c8cc94f57e8",
    "status": "succeeded",
    "params": {
      "pipetteName": "p300_single_gen2",
      "mount": "left"
    },
    "result": {
      "pipetteId": "7912f90b-a97f-455f-b796-27c5806939d3"
    },
    "startedAt": "2025-06-06T05:45:12.430408Z",
    "completedAt": "2025-06-06T05:45:14.353627Z",
    "intent": "setup",
    "notes": []
  }
}


<Response [201]>

## Caution!
* If crash happens/notebook restarts, the code will obtain the labware information from the robot and transfer them back to notebook
* If a previous run already exits, you can "remember" the previous labware set up without needing to start a new run. 

In [ ]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

# Labware Declaration 
* tell the robot that a well plate/Pipette is coming

### opentrons 300 uL (Sterile, for live tumor only)

In [ ]:
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "opentrons_96_tiprack_300ul"
#Load the tip rack. Slot = 1 by default.
# Change the slot number if the Black Pipette tiprack is in different spot
r = openapi.load_labware(TIP_RACK, 11)

ValueError: run_id is not set. Set it before executing commands.

In [ ]:
 # Pick up the pipette tip at a specific location, change the location number if needed
r = openapi.pick_up_tip(openapi.labware_dct['11'], "A1")

### VWR 200 uL XL

In [10]:
custom_labware_path = os.path.join(paths.BASE_DIR,'protocols','vwr_96_tiprack_200ul_xl.json')
with open(custom_labware_path, 'r') as json_file:
    custom_labware = json.load(json_file)

command_dict = {
            "data": custom_labware
        }

command_payload = json.dumps(command_dict)

url = openapi.get_url('runs')+ f'/{openapi.run_id}/'+ 'labware_definitions'
r = requests.post(url = url, headers = openapi.HEADERS, params = {"waitUntilComplete": True}, data = command_payload)
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "vwr_96_tiprack_200ul_xl"
#Load the tip rack. Slot = 1 by default.
openapi.load_labware(TIP_RACK, 10, namespace='custom_beta',verbose=True)

Labware ID:
6cfdae09-1cf4-4b75-a2fb-8818f7598dad



<Response [201]>

* Check if the pipette tiprack is in slot 10, if not, put it in!
* Check the location of the tip, make sure there's a tip in the tiprack

In [40]:
# Use to restore labware and general run information after the notebook crashes
# always check if the tiprack is in slot
r = openapi.pick_up_tip(openapi.labware_dct['10'], "B10")

### Well Plate 

In [12]:
# Putting in a 6 well plate to slot 4
RESERVOIR = "corning_6_wellplate_16.8ml_flat"
r = openapi.load_labware(RESERVOIR, 4, namespace='opentrons',verbose=True)

Labware ID:
60de0012-e610-4876-bbd3-481c9c18286f



In [13]:
# Comment/uncomment to change well plate type
WELL_PLATE = "corning_96_wellplate_360ul_flat"
# WELL_PLATE = "corning_24_wellplate_3.4ml_flat"
# WELL_PLATE = "corning_384_wellplate_112ul_flat"
r = openapi.load_labware(WELL_PLATE, 5, namespace='opentrons',verbose=True)

Offset (0, 0, 64.2) added to run for corning_96_wellplate_360ul_flat in slot 5.
Labware URI:
opentrons/corning_96_wellplate_360ul_flat/1

Check offset before using ...
Labware ID:
3ab2b20e-ede1-4ae1-9241-d2bc3137a87c



* If you want to change different well plate, tell the robot to forget previous well plate using this code first

In [13]:
openapi.move_labware(openapi.labware_dct['5'], new_location='offDeck')

ValueError: run_id is not set. Set it before executing commands.

* telling the robot to pick up liquid and put it in the well plate (only for dead tumor)

In [15]:
openapi.retract_axis('leftZ')

columns = list(range(1,13))

rows = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

x_offset = 0.0
y_offset = 0.0

row_idx = 0
column_idx = 0

source_slot = '4'
dest_slot = '1'
source_well = 'A1'

openapi.blow_out(openapi.labware_dct[source_slot], source_well, well_location='center', flow_rate = 200)
openapi.aspirate(openapi.labware_dct[source_slot], "A2", volume = 50, flow_rate = 200)
openapi.dispense(openapi.labware_dct[source_slot], "A2", volume = 50, flow_rate = 200)
openapi.retract_axis('leftZ')

while row_idx < len(rows):
    row = rows[row_idx]
    while column_idx < len(columns):
        column = columns[column_idx]
        r = openapi.dispense(openapi.labware_dct[dest_slot], f"{row}{column}", well_location = 'bottom', offset = (x_offset, y_offset, 1.0), volume = 100, flow_rate = 200)
        openapi.move_relative('z', 20)
        responce_dict = json.loads(r.text)['data']
        if responce_dict['status'] == 'failed':
            if responce_dict['error']['errorType'] == 'InvalidDispenseVolumeError':
                print('Refilling pipette')
                openapi.aspirate(openapi.labware_dct[source_slot], source_well, well_location ='bottom',offset = (0,0,0.5), volume = 200, flow_rate = 200)
        else:
            column_idx += 1
    column_idx = 0
    row_idx += 1

openapi.blow_out(openapi.labware_dct[source_slot], source_well, well_location='center', flow_rate = 200)
openapi.aspirate(openapi.labware_dct[source_slot], "A2", volume = 50, flow_rate = 200)
openapi.dispense(openapi.labware_dct[source_slot], "A2", volume = 50, flow_rate = 200)
openapi.retract_axis('leftZ')

Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette
Refilling pipette


<Response [201]>

# Robot to Camera calibration
* mapping the two coordinate systems: creating a relation between the pixels of the camera and the real-world coordinates of the robot.
* MAKE SURE the white pad light at the bottom is on!!

In [15]:
# If the lights were on, turn them off using this code
openapi.toggle_lights()

<Response [200]>

* "Aim" the robot: adjust camera focus and robot position in relation to the calibration target
* +/- sign on the keyboard increases the stepsize, up/down/left/right keys can shift the robot's position

In [14]:
squaresX=7
squaresY=5 
squareLength=0.022
markerLength=0.011
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
params = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, params)
board = cv2.aruco.CharucoBoard((squaresX, squaresY), squareLength, markerLength, aruco_dict)
calibration_data = utils.load_calibration_config(calibration_profile)
manual_movement = utils.ManualRobotMovement(openapi)

calib_origin = calibration_data['calib_origin']
openapi.move_to_coordinates(calib_origin, min_z_height=1, verbose=False)

cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)

robot_coords = []
new_calib_origin = None

while True:
 # Capture frame-by-frame
    ret, frame = over_cam.read()
    # frame = frame_ops.undistort_frame(frame)

    x, y, z = openapi.get_position(verbose=False)[0].values()
    (text_width, text_height), _ = cv2.getTextSize(f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
    cv2.rectangle(frame, (10, 0), (10 + text_width, text_height + 100), (0, 0, 0), -1)
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Step size: {manual_movement.step} mm", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    center_screen_x = frame.shape[1] // 2
    center_screen_y = frame.shape[0] // 2
    cv2.circle(frame, (center_screen_x, center_screen_y), 5, (0, 0, 255), -1)
    cv2.putText(frame, f"Center: ({center_screen_x}, {center_screen_y})", (center_screen_x + 10, center_screen_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Calculate the center of each quarter of the screen
    quarter_centers = [
        (center_screen_x // 2, center_screen_y // 2),
        (3 * center_screen_x // 2, center_screen_y // 2),
        (center_screen_x // 2, 3 * center_screen_y // 2),
        (3 * center_screen_x // 2, 3 * center_screen_y // 2)
    ]

    # Draw circles at the center of each quarter
    for qx, qy in quarter_centers:
        cv2.circle(frame, (qx, qy), 5, (0, 255, 255), -1)
        cv2.putText(frame, f"({qx}, {qy})", (qx + 10, qy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    marker_corners, marker_ids, _ = detector.detectMarkers(frame)
    if marker_corners:
        for corner in marker_corners:
            corner = corner.reshape((4, 2))
            for point in corner:
                cv2.circle(frame, tuple(point.astype(int)), 5, (0, 255, 0), -1)

            center_x = int(np.mean(corner[:, 0]))
            center_y = int(np.mean(corner[:, 1]))
            cv2.circle(frame, (center_x, center_y), 5, (255, 0, 0), -1)
            cv2.putText(frame, f"({center_x}, {center_y})", (center_x + 10, center_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # Calculate side lengths
    side_lengths = []
    if marker_corners:
        for corner in marker_corners[0]:
            for i in range(4):
                side_length = np.linalg.norm(corner[i] - corner[(i + 1) % 4])
                side_lengths.append(side_length)

    # Calculate the average side length
        average_side_length = np.mean(side_lengths)
        area = cv2.contourArea(marker_corners[0])
        one_d_ratio = 13.83 / average_side_length
        size_conversion_ratio = 13.83 ** 2 / area
        cv2.putText(frame, f"Area of marker: {area:.2f}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow("video", frame)

    key_pressed = cv2.waitKey(1)

    if key_pressed == ord('q'):
        keyboard.unhook_all()
        x, y, z = openapi.get_position(verbose=False)[0].values()
        new_calib_origin = (x, y, z)
        break

# When everything done, release the capture
# cap.release_camera()
cv2.destroyAllWindows()

calibration_data = utils.load_calibration_config(calibration_profile)
calibration_data['size_conversion_ratio'] = size_conversion_ratio
calibration_data['one_d_ratio'] = one_d_ratio
if new_calib_origin:
    calibration_data['calib_origin'] = new_calib_origin
    print(f"New calibration origin set to: {new_calib_origin}")

utils.save_calibration_config(calibration_profile, calibration_data)

New calibration origin set to: (179.55999999999997, 161.78, 115.00000000000001)



[CONTROL] Emergency stop! Shutting down...

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Emergency stop! Shutting down...

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Emergency stop! Shutting down...

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Emergency stop! Shutting down...

[CONTROL] Robot RESUMED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED

[CONTROL] Emergency stop! Shutting down...

[CONTROL] Robot RESUMED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED

[CONTROL] Robot RESUMED

[CONTROL] Robot PAUSED

[CONTROL] Robot RESUMED


* execute the calibration

In [15]:
calibration_data = utils.load_calibration_config(calibration_profile)
calib_origin = calibration_data['calib_origin']

spacing = 5  # Distance from the calib_point in mm

# Calculate the four coordinates
calibration_points = [
    (calib_origin[0] + spacing, calib_origin[1] + spacing),  # Right
    (calib_origin[0] + spacing, calib_origin[1] - spacing),  # Left
    (calib_origin[0] - spacing, calib_origin[1] - spacing),  # Up
    (calib_origin[0] - spacing, calib_origin[1] + spacing)   # Down
]

robot_coords = []
camera_coords = []


# window = cap.get_window()
cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)

for calib_pt in calibration_points:
    openapi.move_to_coordinates((*calib_pt, 100), min_z_height=1, verbose=False)
    cv2.waitKey(1000)
    # frame = cap.get_frame(undist=True)
    ret, frame = over_cam.read()
    frame = frame_ops.undistort_frame(frame)
    
    x, y, z = openapi.get_position(verbose=False)[0].values()
    (text_width, text_height), _ = cv2.getTextSize(f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
    cv2.rectangle(frame, (10, 0), (10 + text_width, text_height + 70), (0, 0, 0), -1)
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Step size: {manual_movement.step} mm", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    center_screen_x = frame.shape[1] // 2
    center_screen_y = frame.shape[0] // 2
    cv2.circle(frame, (center_screen_x, center_screen_y), 5, (0, 0, 255), -1)
    cv2.putText(frame, f"Center: ({center_screen_x}, {center_screen_y})", (center_screen_x + 10, center_screen_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Calculate the center of each quarter of the screen
    quarter_centers = [
        (center_screen_x // 2, center_screen_y // 2),
        (3 * center_screen_x // 2, center_screen_y // 2),
        (center_screen_x // 2, 3 * center_screen_y // 2),
        (3 * center_screen_x // 2, 3 * center_screen_y // 2)
    ]

    # Draw circles at the center of each quarter
    for qx, qy in quarter_centers:
        cv2.circle(frame, (qx, qy), 5, (0, 255, 255), -1)
        cv2.putText(frame, f"({qx}, {qy})", (qx + 10, qy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    marker_corners, marker_ids, _ = detector.detectMarkers(frame)
    if marker_corners:
        for corner in marker_corners:
            corner = corner.reshape((4, 2))
            for point in corner:
                cv2.circle(frame, tuple(point.astype(int)), 5, (0, 255, 0), -1)

            center_x = int(np.mean(corner[:, 0]))
            center_y = int(np.mean(corner[:, 1]))
            cv2.circle(frame, (center_x, center_y), 5, (255, 0, 0), -1)
            cv2.putText(frame, f"({center_x}, {center_y})", (center_x + 10, center_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    cv2.waitKey(1)
    cv2.imshow("video", frame)
    x, y, z = openapi.get_position(verbose=False)[0].values()
    robot_coords.append((x, y))
    camera_coords.append((center_x, center_y))

cv2.destroyAllWindows()

calibration_data = utils.load_calibration_config(calibration_profile)

camera_coords = utils.sort_coordinates(camera_coords)
robot_coords = utils.sort_coordinates(robot_coords, reverse_y=True)

robot_to_camera_coords = {tuple(robot_coord): tuple(camera_coord) for robot_coord, camera_coord in zip(robot_coords, camera_coords)}
tf_mtx = utils.compute_tf_mtx(robot_to_camera_coords)

calibration_data['tf_mtx'] = tf_mtx.tolist()

utils.save_calibration_config(calibration_profile, calibration_data)

# Pipette offset calibration

In [41]:
# Need the lights on for this process
openapi.toggle_lights()

<Response [200]>

* excute automatic pipette calibration
* align crosshair to center of the image and press q for quit
* manually change the location of the crosshair disk to ensure it's center aligns with the camera

In [42]:
path = os.path.join(paths.BASE_DIR, 'outputs', 'images', 'target_template_2.png')
template = cv2.imread(path, 0)  # Replace 'template.png' with your template image path
template_height, template_width = template.shape[:2]
calibration_data = utils.load_calibration_config(calibration_profile)
tf_mtx = np.array(calibration_data['tf_mtx'])
calib_origin = np.array(calibration_data['calib_origin'])[:2]
tip_calib_origin = np.array(calibration_data['tip_calib_origin'])
offset = np.array(calibration_data['offset'])
model_path = os.path.join(paths.ML_MODELS_DIR,'tip_detector_v1.pt')
model = YOLO(model_path)


#Settings
calib_module_height = 69  # Height for the pipette offset calibration module
detection_offset_x = -4
detection_offset_y = 0  # Offset to apply to the detected coordinates
#Move to pipette offset calibration module:
openapi.move_to_coordinates(tip_calib_origin, min_z_height=calib_module_height-0.1, verbose=False)

cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)

manual_movement = utils.ManualRobotMovement(openapi)
new_tip_calib_origin = None
while True:
 # Capture frame-by-frame
    ret, frame = over_cam.read()
    frame = frame_ops.undistort_frame(frame)

    x, y, z = openapi.get_position(verbose=False)[0].values()
    (text_width, text_height), _ = cv2.getTextSize(f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
    cv2.rectangle(frame, (10, 0), (10 + text_width, text_height + 100), (0, 0, 0), -1)
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Step size: {manual_movement.step} mm", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    center_screen_x = frame.shape[1] // 2
    center_screen_y = frame.shape[0] // 2
    cv2.circle(frame, (center_screen_x, center_screen_y), 5, (0, 0, 255), -1)
    cv2.putText(frame, f"Center: ({center_screen_x}, {center_screen_y})", (center_screen_x + 10, center_screen_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    cv2.imshow("video", frame)
    key_pressed = cv2.waitKey(1)

    if key_pressed == ord('q'):
        keyboard.unhook_all()
        x, y, z = openapi.get_position(verbose=False)[0].values()
        new_tip_calib_origin = (x, y, z)
        break

cv2.destroyAllWindows()


ret, frame = over_cam.read()
assert ret, "Failed to capture frame from over_cam."
frame = frame_ops.undistort_frame(frame)
image = frame.copy()
plot_img_over = frame.copy()
image = image[..., ::-1]  # Convert BGR to RGB as YOLO expects RGB input
results = model.predict(
        source=image,  # Now pointing to a directory instead of a single file
        conf=0.25,         # Confidence threshold
        save=False,         # Save the annotated images
        save_txt=False,    # Save YOLO-format prediction labels (optional)
        show=False,         # Show images in pop-up windows (if GUI available)
        imgsz=2016,
        verbose = False               # Ensure inference matches your training resolution
    )

image_center = (image.shape[1] // 2, image.shape[0] // 2)
data = []
for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
        data.append({'class': model.names[cls], 'confidence': conf, 'center_x': center_x, 'center_y': center_y})

        if model.names[cls] == 'point':
            color = (0, 255, 0)  # Green for points
        else:
            color = (0, 0, 255)  # Red for tips
            
        # Draw rectangle around the detected object
        cv2.rectangle(plot_img_over, (x1, y1), (x2, y2), color, 2)

        # Add label with class name and confidence score
        label = f"{model.names[cls]} {conf:.2f}"
        cv2.putText(plot_img_over, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Draw center point of detection
        cv2.circle(plot_img_over, (center_x, center_y), 4, color, -1)

# Select the point closest to the center of the image
if data:
    closest_obj = min(
        (obj for obj in data if obj['class'] == 'point'),
        key=lambda obj: (obj['center_x'] - image_center[0]) ** 2 + (obj['center_y'] - image_center[1]) ** 2,
        default=None
    )
    if closest_obj is not None:
        cv2.circle(frame, (closest_obj['center_x'], closest_obj['center_y']), 8, (0, 255, 255), 2)
        cv2.putText(frame, "Closest", (closest_obj['center_x'] + 10, closest_obj['center_y'] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
assert closest_obj is not None, "No 'point' class detected in the image."
crosshair_x, crosshair_y = closest_obj['center_x'], closest_obj['center_y']

#Move to the center of the detected template
X_init, Y_init, _ = tf_mtx @ (crosshair_x, crosshair_y, 1)
print('init:', X_init, Y_init)

x, y, _ = openapi.get_position(verbose=False)[0].values()
diff = np.array([x,y]) - np.array(calibration_data['calib_origin'])[:2]
X = X_init + diff[0] + offset[0]
Y = Y_init + diff[1] + offset[1]

print(f"Robot coords: ({x}, {y})")
print(f"Clicked on: ({X}, {Y})")
openapi.move_to_coordinates((X + detection_offset_x, Y + detection_offset_y, calib_module_height), min_z_height=calib_module_height-0.1, verbose=False)
time.sleep(1)

ret, frame = under_cam.read()
assert ret, "Failed to capture frame from under_cam."
frame = frame[..., ::-1]  # Convert BGR to RGB as YOLO expects RGB input
results = model.predict(
        source=frame,  # Now pointing to a directory instead of a single file
        conf=0.25,         # Confidence threshold
        save=False,         # Save the annotated images
        save_txt=False,    # Save YOLO-format prediction labels (optional)
        show=False,         # Show images in pop-up windows (if GUI available)
        imgsz=2016,
        verbose = False               # Ensure inference matches your training resolution
    )

image = frame.copy()
plot_img = frame.copy()
image_center = (image.shape[1] // 2, image.shape[0] // 2)
data = []
for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
        data.append({'class': model.names[cls], 'confidence': conf, 'center_x': center_x, 'center_y': center_y})
        # Draw bounding boxes and labels on the plot image
        if model.names[cls] == 'point':
            color = (0, 255, 0)  # Green for points
        else:
            color = (0, 0, 255)  # Red for tips
            
        # Draw rectangle around the detected object
        cv2.rectangle(plot_img, (x1, y1), (x2, y2), color, 2)

        # Add label with class name and confidence score
        label = f"{model.names[cls]} {conf:.2f}"
        cv2.putText(plot_img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # Draw center point of detection
        cv2.circle(plot_img, (center_x, center_y), 4, color, -1)

# Create a dataframe
df = pd.DataFrame(data)

# Calculate the distance of each "point" to the center of the frame
df['distance_to_center'] = ((df['center_x'] - image_center[0]) ** 2 + (df['center_y'] - image_center[1]) ** 2) ** 0.5

# Identify the closest "point" to the center of the frame
df['is_closest_to_center'] = (df['class'] == 'point') & (df['distance_to_center'] == df.loc[df['class'] == 'point', 'distance_to_center'].min()) & (df['distance_to_center'] < 100)

if df['is_closest_to_center'].any():
    assert 'tip' in df['class'].values and 'point' in df['class'].values, "Both 'tip' and 'point' classes must be present in the dataframe."
    distances_from_closest = np.sqrt(
        (df.loc[df['is_closest_to_center'] & (df['class'] == 'point'), 'center_x'].values[0] - df.loc[df['class'] == 'point', 'center_x'])**2 +
        (df.loc[df['is_closest_to_center'] & (df['class'] == 'point'), 'center_y'].values[0] - df.loc[df['class'] == 'point', 'center_y'])**2
    )
    distances_from_closest = distances_from_closest[distances_from_closest > 0]
    linear_distance_ratio = 20.25 / np.mean(distances_from_closest)
    print(distances_from_closest, linear_distance_ratio)
        # Calculate the distance from the center-most "point" class to the "tip" class
    center_point_coords = df.loc[df['is_closest_to_center'], ['center_x', 'center_y']].values[0]
    tip_coords = df.loc[df['class'] == 'tip', ['center_x', 'center_y']].values[0]

    x_dist_to_tip = center_point_coords[0] - tip_coords[0]
    y_dist_to_tip = center_point_coords[1] - tip_coords[1]
    print(x_dist_to_tip, y_dist_to_tip)

    y_dist_to_tip_mm = x_dist_to_tip * linear_distance_ratio
    x_dist_to_tip_mm = y_dist_to_tip * linear_distance_ratio
    print(f"x_dist_to_tip_mm: {x_dist_to_tip_mm}, y_dist_to_tip_mm: {y_dist_to_tip_mm}")

assert (x_dist_to_tip_mm is not None) and (y_dist_to_tip_mm is not None), "Failed to calculate distances to tip."
actual_offset_x = x_dist_to_tip_mm + detection_offset_x
actual_offset_y = y_dist_to_tip_mm + detection_offset_y
assert abs(actual_offset_x) < 40 and abs(actual_offset_y) < 40, "Offsets are too large, please check the calibration."
openapi.move_relative('x', x_dist_to_tip_mm, verbose=False)
openapi.move_relative('y', y_dist_to_tip_mm, verbose=False)

# Take one frame from the under camera and display it for verification
time.sleep(1)
for _ in range(5):
    ret, verification_frame = under_cam.read()
assert ret, "Failed to capture verification frame from under_cam."

cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)
cv2.imshow("video", verification_frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.namedWindow("detection", cv2.WINDOW_NORMAL)
cv2.resizeWindow("detection", 1348, 1011)
cv2.imshow("detection", plot_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.namedWindow("detection_over", cv2.WINDOW_NORMAL)
cv2.resizeWindow("detection_over", 1348, 1011)
cv2.imshow("detection_over", plot_img_over)
cv2.waitKey(0)
cv2.destroyAllWindows()

print(f"Actual offset applied: ({actual_offset_x}, {actual_offset_y}) mm")
x, y, _ = openapi.get_position(verbose=False)[0].values()
calibration_data['offset'] = [x-(X_init+diff[0]), y-(Y_init+diff[1])]
calibration_data['tip_calib_origin'] = new_tip_calib_origin
utils.save_calibration_config(calibration_profile, calibration_data)

openapi.retract_axis('leftZ')

init: 181.3407228408677 161.68360894425928
Robot coords: (282.19999999999993, 161.27)
Clicked on: (298.1247150113912, 222.71353016337386)
1    817.352433
2    811.355039
3    812.271506
4    813.354166
dtype: float64 0.024889891840495084
-11 188
x_dist_to_tip_mm: 4.679299666013076, y_dist_to_tip_mm: -0.27378881024544593
Actual offset applied: (0.6792996660130761, -0.27378881024544593) mm


<Response [201]>

# Picking procedure

* Declare the function

In [20]:
class RobotState(Enum):
    IDLE = 'idle'
    CAPTURE_FRAME = 'capture_frame'
    ANALYZE_FRAME = 'analyze_frame'
    APPROACH_TARGET = 'approach_target'
    PICKUP_SAMPLE = 'pickup_sample'
    VERIFY_PICKUP = 'verify_pickup'
    DEPOSIT_BACK = 'deposit_liquid_back'
    TRANSFER_TO_WELL = 'transfer_to_well'
    AUTO_SHAKE = 'auto_shake'
    PAUSED = 'paused'
    CANCELED = 'canceled'
    COMPLETED = 'completed'

try:
    import keyboard
    KEYBOARD_AVAILABLE = True
except ImportError:
    KEYBOARD_AVAILABLE = False
    print("Warning: 'keyboard' module not found. Install with: pip install keyboard")

class TissuePickerFSM():
    def __init__(self, config: core.PickingConfig, routine: core.Routine, logger: core.MarkdownLogger):
        self.state = RobotState.IDLE
        self.display_process = camera.FrameDisplayProcess("Tissue Picker Vision")
        self.logger = logger
        self.running = True
        self.paused = False
        self.keyboard_lock = threading.Lock()
        self.keyboard_hooks = []

        self.config = config
        self.routine = routine
        self.cr = core.Core()

        self.calibration_data = utils.load_calibration_config(calibration_profile)
        self.tf_mtx = np.array(self.calibration_data['tf_mtx'])
        self.calib_origin = np.array(self.calibration_data['calib_origin'])[:2]
        self.offset = np.array(self.calibration_data['offset'])
        self.size_conversion_ratio = self.calibration_data['size_conversion_ratio']
        self.one_d_ratio = self.calibration_data['one_d_ratio']

        self.isolated = []
        self.pickable_cuboids = []
        self.world_coordinates = []
        self.cuboid_choice = None
        self.current_frame = None
        self.current_well = self.routine.get_next_well()

    def calculate_robot_coordinates(self, cX, cY, robot_x, robot_y):
        X_init, Y_init, _ = self.tf_mtx @ (cX, cY, 1)
        diff = np.array([robot_x,robot_y]) - self.calib_origin
        X = X_init + diff[0] + self.offset[0]
        Y = Y_init + diff[1] + self.offset[1]
        return X, Y

    def cv_pipeline(self, frame):
        mask = np.zeros_like(frame, dtype=np.uint8)
        cv2.circle(mask, self.config.circle_center, self.config.circle_radius + int(self.config.minimum_distance / self.one_d_ratio), (255, 255, 255), -1)
        masked_frame = cv2.bitwise_and(frame, mask)

        gray = cv2.cvtColor(masked_frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (11, 11), 0) #was (11, 11)
        thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,61,3) #was 4
        self.bubble_thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,61,3)
        # self.bubble_thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,35,5)

        kernel = np.ones((3,3),np.uint8)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
        mask_inv = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        thresh = cv2.bitwise_and(thresh, mask_inv)

        # Find contours in the masked frame
        contours, hei = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        filtered_contours = [contour for contour in contours if self.config.contour_filter_window[0] < cv2.contourArea(contour) < self.config.contour_filter_window[1]]
        self.cr.cuboids = filtered_contours
        self.cr.cuboid_dataframe(self.cr.cuboids)

        cuboid_size_micron2 = self.cr.cuboid_df.area * self.size_conversion_ratio * 10e5
        cuboid_diameter = 2 * np.sqrt(cuboid_size_micron2 / np.pi)
        dist_mm = self.cr.cuboid_df.min_dist * self.one_d_ratio
        self.cr.cuboid_df['diameter_microns'] = cuboid_diameter
        self.cr.cuboid_df['min_dist_mm'] = dist_mm
        # Check if the dataframe is not empty before applying operations
        if len(self.cr.cuboid_df) > 0:
            self.cr.cuboid_df['bubble'] = self.cr.cuboid_df.apply(lambda row: not bool(self.bubble_thresh[int(row['cY']), int(row['cX'])]), axis=1)

            # Filter out elongated contours
            self.pickable_cuboids = self.cr.cuboid_df.loc[((self.config.cuboid_size_theshold[0] < self.cr.cuboid_df['diameter_microns']) & 
                                                (self.cr.cuboid_df['diameter_microns'] < self.config.cuboid_size_theshold[1])) &
                                                ((self.cr.cuboid_df['aspect_ratio'] > self.config.aspect_ratio_window[0]) & 
                                                    (self.cr.cuboid_df['aspect_ratio'] < self.config.aspect_ratio_window[1])) &
                                                ((self.cr.cuboid_df['circularity'] > self.config.circularity_window[0]) &
                                                    (self.cr.cuboid_df['circularity'] < self.config.circularity_window[1])) & 
                                                (self.cr.cuboid_df['bubble'] != True)].copy()

            # Check if cuboid centers are within the circle radius from the current circle center
            self.pickable_cuboids['distance_to_center'] = self.pickable_cuboids.apply(
                lambda row: np.sqrt((row['cX'] - self.config.circle_center[0])**2 + (row['cY'] - self.config.circle_center[1])**2), axis=1
            )
            self.pickable_cuboids = self.pickable_cuboids[self.pickable_cuboids['distance_to_center'] <= self.config.circle_radius]
            self.isolated = self.pickable_cuboids.loc[self.pickable_cuboids.min_dist_mm > self.config.minimum_distance]
        else:
            self.pickable_cuboids = []
            self.isolated = []

    def draw_annotations(self, frame):
        cv2.circle(frame, self.config.circle_center, self.config.circle_radius + int(self.config.minimum_distance / self.one_d_ratio), (0, 0, 255), 2)
        cv2.circle(frame, self.config.circle_center, self.config.circle_radius, (0, 255, 0), 2)
        # Add a black rectangle behind the text
        text_background_height = 250  # Adjust height to fit all text lines
        text_background_width = 320  # Full width of the frame
        cv2.rectangle(frame, (0, 0), (text_background_width, text_background_height), (0, 0, 0), -1)
        if self.routine.current_well is None:
            self.routine.get_next_well()
        cv2.putText(frame, f"Filling well: {self.routine.current_well}", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        if self.paused:
            cv2.putText(frame, "Paused", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        if self.cr.cuboids:
            cv2.drawContours(frame, self.cr.cuboids, -1, (0, 0, 255), 2)
            cv2.drawContours(frame, self.pickable_cuboids.contour.values.tolist(), -1, (0, 255, 255), 2)
            cv2.drawContours(frame, self.isolated.contour.values.tolist(), -1, (0, 255, 0), 2)
            if hasattr(self.cr.cuboid_df, 'bubble'):
                bubble_contours = self.cr.cuboid_df[self.cr.cuboid_df['bubble']].contour.values.tolist()
                if bubble_contours:
                    cv2.drawContours(frame, bubble_contours, -1, (255, 0, 255), 2)  # Purple color for bubbles
        cv2.putText(frame, f"# Objects: {len(self.cr.cuboids)}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"# Pickable: {len(self.pickable_cuboids)}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"# Isolated: {len(self.isolated)}", (10, 190), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cuboids_in_size_range = self.cr.cuboid_df.loc[(self.config.cuboid_size_theshold[0] < self.cr.cuboid_df['diameter_microns']) & 
                                        (self.cr.cuboid_df['diameter_microns'] < self.config.cuboid_size_theshold[1])].copy()
        cv2.putText(frame, f"# In size range: {len(cuboids_in_size_range)}", (10, 230), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        if self.cuboid_choice is not None:
            for idx, row in self.cuboid_choice.iterrows():
                rect = cv2.minAreaRect(row['contour'])  # ((center_x, center_y), (width, height), angle)
                box = cv2.boxPoints(rect)
                box = np.intp(box)  # Convert to integer coordinates
                cv2.drawContours(frame, [box], 0, (0, 0, 0), 2)
            # for idx, row in self.cuboid_choice.iterrows():
                # x, y, w, h = cv2.boundingRect(row['contour'])
                # # Calculate center and size of the bounding box
                # center_x = x + w / 2
                # center_y = y + h / 2
                # new_w = int(w * 1.25)
                # new_h = int(h * 1.25)
                # new_x = int(center_x - new_w / 2)
                # new_y = int(center_y - new_h / 2)
                # cv2.rectangle(frame, (new_x, new_y), (new_x + new_w, new_y + new_h), (0, 0, 0), 2)

        return frame
    
    def _setup_keyboard_hooks(self):
        """Setup global keyboard hooks using the keyboard module"""
        if not KEYBOARD_AVAILABLE:
            print("Keyboard module not available. No keyboard controls will work.")
            return
            
        def on_pause_key(event):
            if event.event_type == keyboard.KEY_DOWN:
                with self.keyboard_lock:
                    self.paused = not self.paused
                    status = "PAUSED" if self.paused else "RESUMED"
                    print(f"\n[CONTROL] Robot {status}")
        
        def on_escape_key(event):
            if event.event_type == keyboard.KEY_DOWN:
                with self.keyboard_lock:
                    print("\n[CONTROL] Emergency stop! Shutting down...")
                    self.running = False
        
        # Register keyboard hooks
        try:
            self.keyboard_hooks.append(keyboard.on_press_key('p', on_pause_key))
            self.keyboard_hooks.append(keyboard.on_press_key('esc', on_escape_key))
            print("Keyboard hooks registered successfully!")
        except Exception as e:
            print(f"Failed to register keyboard hooks: {e}")
            print("Note: On some systems, you may need to run as administrator/root.")
    
    def _cleanup_keyboard_hooks(self):
        """Remove keyboard hooks"""
        if KEYBOARD_AVAILABLE:
            try:
                for hook in self.keyboard_hooks:
                    keyboard.unhook(hook)
                self.keyboard_hooks.clear()
            except Exception as e:
                print(f"Error cleaning up keyboard hooks: {e}")
    
    def start(self):
        """Start the FSM with global keyboard controls"""
        print("=== Tissue Picker Robot FSM Started ===")
        if KEYBOARD_AVAILABLE:
            print("  'p' - Pause/Resume")
            print("  'ESC' - Emergency stop and quit")
        else:
            print("No keyboard controls available (keyboard module not installed)")
        print("=" * 60)
        
        # Setup keyboard hooks
        self._setup_keyboard_hooks()
        self.display_process.start()
        
        try:
            while self.running:
                with self.keyboard_lock:
                    if self.paused:
                        time.sleep(0.1)
                        continue
                
                # Execute current state
                if hasattr(self, f"state_{self.state.value}"):
                    getattr(self, f"state_{self.state.value}")()
                else:
                    print(f"Error: Unknown state {self.state.value}")
                    self.running = False
                
                # Small delay to prevent overwhelming the output
                time.sleep(0.1)
        
        except KeyboardInterrupt:
            print("\n[CONTROL] Keyboard interrupt received. Shutting down...")
            self.running = False
        
        finally:
            # Cleanup keyboard hooks
            self._cleanup_keyboard_hooks()
            self.display_process.stop()
            cv2.destroyAllWindows()
            print("\n=== Tissue Picker Robot FSM Stopped ===")

    def state_idle(self):
        print("Robot is idle. Press 'p' to continue...")
        self.paused = True  # Ensure the robot is paused in idle state
        # Move to picking position
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((self.calib_origin[0],self.calib_origin[1],115), min_z_height=self.config.dish_bottom, verbose=False)
        # Turn off lights
        current_status = openapi.get("lights", openapi.HEADERS)
        current_status = json.loads(current_status.text)
        is_on = current_status['on']
        if is_on:
            openapi.toggle_lights()
        self.display_process.send_status("Robot IDLE - Press 'p' to continue", (0, 0, 255))

        while self.paused and self.running:
            try:
                ret, frame = over_cam.read()
                frame = frame_ops.undistort_frame(frame)
                plot_frame = frame.copy()
                self.cv_pipeline(frame)
                self.draw_annotations(plot_frame)
                self.display_process.send_frame(plot_frame)
            except Exception as e:
                print(f"Error in idle state: {e}")
                self.running = False
                break

        self.state = RobotState.CAPTURE_FRAME
        self.start_time = time.time()

    def state_capture_frame(self):
        openapi.move_to_coordinates((self.calib_origin[0],self.calib_origin[1],115), min_z_height=self.config.dish_bottom, verbose=False)
        time.sleep(0.5)

        ret, frame = over_cam.read()
        self.current_frame = frame_ops.undistort_frame(frame)
        self.state = RobotState.ANALYZE_FRAME

    def state_auto_shake(self):
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((235, 223, 65), verbose=False)
        for _ in range(3):
            openapi.move_relative('x', -10)
            openapi.move_relative('x', 10)
            time.sleep(0.5)
        openapi.move_relative('x', 2)
        time.sleep(0.5)
        openapi.move_relative('x', -2)
        openapi.retract_axis('leftZ')
        time.sleep(2)
        self.state = RobotState.CAPTURE_FRAME

    def state_analyze_frame(self):
        self.cv_pipeline(self.current_frame)
        if len(self.isolated) == 0:
            print("No isolated cuboids found. Pausing...")
            self.logger.log("No cuboids found in the selected region. Pausing...")
            self.state = RobotState.AUTO_SHAKE
            return

        next_well = self.routine.get_next_well()
        cuboids_to_fill = self.routine.well_plan[next_well] - self.routine.filled_wells[next_well]
        if not self.config.one_by_one:
            if len(self.isolated) > cuboids_to_fill:
                self.cuboid_choice = self.isolated.sample(n=cuboids_to_fill)
            else:
                self.cuboid_choice = self.isolated
        else:
            self.cuboid_choice = self.isolated.sample(n=1)

        self.logger.log_table(self.cuboid_choice.loc[:, self.cuboid_choice.columns != 'contour'], title=f"Filling well {next_well}")
        plot_frame = self.current_frame.copy()
        self.draw_annotations(plot_frame)
        self.display_process.send_frame(plot_frame)
        
        self.state = RobotState.APPROACH_TARGET

    def state_approach_target(self):
        cuboid_coordinates = self.cuboid_choice[['cX', 'cY']].values
        self.world_coordinates = []
        for cX, cY in cuboid_coordinates:
            X, Y = self.calculate_robot_coordinates(cX, cY, self.calib_origin[0], self.calib_origin[1])
            self.world_coordinates.append((X, Y))
        self.state = RobotState.PICKUP_SAMPLE

    def state_pickup_sample(self):
        for x,y in self.world_coordinates:
            openapi.move_to_coordinates((x, y, self.config.pickup_height+20), min_z_height=self.config.dish_bottom, verbose=False, force_direct=True)
            openapi.move_to_coordinates((x, y, self.config.pickup_height), min_z_height=self.config.dish_bottom, verbose=False, force_direct=True)
            openapi.aspirate_in_place(flow_rate = self.config.flow_rate, volume = self.config.vol)
            openapi.move_relative('z', 20)

        self.state = RobotState.VERIFY_PICKUP

    def state_verify_pickup(self):
        openapi.move_to_coordinates((self.calib_origin[0],self.calib_origin[1],115), min_z_height=self.config.dish_bottom, verbose=False, force_direct=True)
        time.sleep(0.75)
        ret, frame = over_cam.read()
        self.current_frame = frame_ops.undistort_frame(frame)
        self.cv_pipeline(self.current_frame)

        plot_frame = self.current_frame.copy()
        self.draw_annotations(plot_frame)
        miss_occurred = False
        if self.cuboid_choice is not None:
            for prev_x, prev_y in self.cuboid_choice[['cX', 'cY']].values:
                cv2.circle(plot_frame, (int(prev_x), int(prev_y)), int(round(self.config.failure_threshold / self.one_d_ratio)), (255, 0, 0), 2)
                check_miss_df = self.pickable_cuboids.copy()
                distances = check_miss_df.apply(lambda row: np.sqrt((row['cX'] - prev_x)**2 + (row['cY'] - prev_y)**2), axis=1).to_numpy()
                distances *= self.one_d_ratio
                if any(distances <= self.config.failure_threshold):
                    print(f"Miss detected at well {self.routine.current_well}.")
                    self.routine.update_well(success=False)
                    miss_occurred = True
                    self.logger.log(f"Miss detected at well {self.routine.current_well}.")

            if not miss_occurred:
                for _, _ in self.cuboid_choice[['cX', 'cY']].values:
                    self.routine.update_well(success=True)

        if miss_occurred:
            self.state = RobotState.DEPOSIT_BACK
        else:
            self.state = RobotState.TRANSFER_TO_WELL

        self.display_process.send_frame(plot_frame)
        

    def state_deposit_liquid_back(self):
        x,y = self.world_coordinates[0]  # Use the first coordinate for depositing back
        openapi.move_to_coordinates((x, y, self.config.pickup_height+20), min_z_height=self.config.dish_bottom, verbose=False, force_direct=True)
        openapi.move_to_coordinates((x, y, self.config.pickup_height+0.5), min_z_height=self.config.dish_bottom, verbose=False, force_direct=True)
        openapi.dispense_in_place(flow_rate = self.config.flow_rate, volume = self.config.vol * len(self.world_coordinates))
        time.sleep(0.5)
        openapi.move_relative('z', 20)

        self.state = RobotState.CAPTURE_FRAME

    def state_transfer_to_well(self):

        openapi.move_to_well(openapi.labware_dct[str(self.config.destination_slot)], self.current_well, 
                             well_location='top', 
                             offset=(self.config.well_offset_x, self.config.well_offset_y, 5), 
                             verbose = False, 
                             force_direct = True)
        openapi.dispense(openapi.labware_dct[str(self.config.destination_slot)], self.current_well, 
                         well_location='bottom', 
                         offset=(self.config.well_offset_x, self.config.well_offset_y, self.config.deposit_offset_z), 
                         volume = self.config.vol * len(self.world_coordinates), 
                         flow_rate = self.config.flow_rate)
        
        time.sleep(self.config.wait_time_after_deposit)
        
        openapi.move_to_well(openapi.labware_dct[str(self.config.destination_slot)], self.current_well, 
                             well_location='top', 
                             offset=(self.config.well_offset_x, self.config.well_offset_y, 5), 
                             verbose=False)
        openapi.move_to_coordinates((self.calib_origin[0],self.calib_origin[1],115), 
                                    min_z_height=self.config.dish_bottom, 
                                    verbose=False, 
                                    force_direct=True)

        self.current_well = self.routine.get_next_well() # Check if the routine is done
        if self.routine.is_done():
            self.state = RobotState.COMPLETED
        else:
            self.state = RobotState.CAPTURE_FRAME

    def state_completed(self):
        self.end_time = time.time()
        print("Process completed successfully.")
        self.logger.log("Picking procedure finished.")
        cv2.destroyAllWindows()
        self.running = False

* Adjust picking settings

In [ ]:
picking_settings = {'vol': 10.0,    # volume in uL, suction when picking up the cuboid
                    'dish_bottom': 65.9, # height of the dish bottom from the robot base in mm
                    'pickup_offset': 0.5,    #how far the tip is from the bottom of the dish when picking up the cuboid
                    'flow_rate': 50.0, # flow rate for both aspiration and dispensing in uL/s
                    'cuboid_size_threshold': (350, 450), #size range in between 350-450 microns diameter, robot filters out cuboids not in this range
                    'failure_threshold': 0.6, #if after pickup there is a cuboid within 0.6mm of the previous location, it is considered a miss
                    'minimum_distance': 1.8, #minimum distance in mm as radius(not diameter) between cuboids to be considered isolated
                    'wait_time_after_deposit': 0.3, #time to wait after depositing the liquid in the well plate in seconds
                    'one_by_one': False,  #true: pick up one cuboid at a time, false: pick up as many as needed to fill the well
                    'well_offset_x': -1.0, #offset in x direction from the center of the well, in mm
                    'well_offset_y': 0.0, # offset in y direction from the center of the well, in mm
                    'deposit_offset_z': 1.75, #offset in z direction from the bottom of the well when depositing the liquid in mm
                    'destination_slot': 5, 
                    'circle_center': (1296, 972), # center of the circular picking area in pixels (x, y), ignore the area outside of this circle in the image
                    'circle_radius': 900,
                    'contour_filter_window': (50, 3000), #min and max area of the contour to be considered, filter out contours outside this range
                    'aspect_ratio_window': (0.75, 1.25), #min and max aspect ratio of the contour to be considered, filter out contours outside this range
                    'circularity_window': (0.40, 0.8)} #measure the circularity of the contour, filter out contours outside this range

picking_config = core.PickingConfig.from_dict(picking_settings)

* fill the tip with some solution

In [49]:
openapi.retract_axis('leftZ')
openapi.aspirate(openapi.labware_dct['4'], "A1", well_location = 'bottom', volume = 20, flow_rate = 50)
openapi.retract_axis('leftZ')

<Response [201]>

* See if you need to adjust x and y offset, if you do, change the offset from the previous picking settings
* make sure the tip is in the center of the well

In [26]:
openapi.move_to_well(openapi.labware_dct['5'], "H1", well_location = 'top', offset=(-1,0,1))

<Response [201]>

* create a plate map

In [60]:
plate_type = 96
dest = core.Destination(plate_type)
well_df = core.create_well_plan(plate_type)
well_df.loc['E':'G', 2:10] = 4
#well_df.loc['A', 1:12] = 1


well_plan = {f"{row}{col}": well_df.loc[row, col] for row in well_df.index for col in well_df.columns if well_df.loc[row, col] > 0}
well_df

,1,2,3,4,5,6,7,8,9,10,11,12
A,0,0,0,0,0,0,0,0,0,0,0,0
B,0,0,0,0,0,0,0,0,0,0,0,0
C,0,0,0,0,0,0,0,0,0,0,0,0
D,0,0,0,0,0,0,0,0,0,0,0,0
E,0,4,4,4,4,4,4,4,4,4,0,0
F,0,4,4,4,4,4,4,4,4,4,0,0
G,0,4,4,4,4,4,4,4,4,4,0,0
H,0,0,0,0,0,0,0,0,0,0,0,0


In [63]:
routine = core.Routine(dest, well_plan, fill_strategy="horizontal")
# routine = core.Routine(dest, well_plan, fill_strategy="vertical")


In [64]:
logger = core.MarkdownLogger(experiment_name="10-15-25_cryopreserve", settings=picking_settings, well_plate=well_df)
logger.log_section("Execution start:")
fsm = TissuePickerFSM(picking_config, routine, logger)
fsm.start()

=== Tissue Picker Robot FSM Started ===
  'p' - Pause/Resume
  'ESC' - Emergency stop and quit
Keyboard hooks registered successfully!
Display process started with PID: 8992
Robot is idle. Press 'p' to continue...
No isolated cuboids found. Pausing...
Miss detected at well F2.
No isolated cuboids found. Pausing...
No isolated cuboids found. Pausing...
No isolated cuboids found. Pausing...
Process completed successfully.

=== Tissue Picker Robot FSM Stopped ===


# Misc Commands

In [38]:
openapi.blow_out_in_place(flow_rate = 50)

<Response [201]>

In [39]:
openapi.drop_tip_in_place()

<Response [201]>

In [43]:
manual_movement = utils.ManualRobotMovement(openapi)

In [44]:
keyboard.unhook_all()

In [45]:
openapi.get_position()

{'x': 193.79980550227927, 'y': 222.4371300583493, 'z': 65.90000000000002}


({'x': 193.79980550227927, 'y': 222.4371300583493, 'z': 65.90000000000002},
 <Response [201]>)

In [47]:
openapi.move_relative('z', 20)

<Response [201]>